In [1]:
import os
import time
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import copy

from resnet_torch import ResNet18

In [2]:
%load_ext autoreload   

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cuda


In [4]:
float_model = ResNet18()

state_dict = torch.load('resnet18_cifar10.pth', map_location=torch.device('cpu'))
float_model.load_state_dict(state_dict)
float_model.to(device)
print()

In [5]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    size_mb = os.path.getsize("temp.p") / 1e6
    print(f'Model Size: {size_mb:.2f} MB')
    os.remove('temp.p')

In [6]:
print_size_of_model(float_model)

Model Size: 44.77 MB


In [7]:
def test(model, data_loader, device_override=None):
    model.eval()
    correct = 0
    total = 0
    
    # Use override device if provided (e.g., forcing CPU for quantized model)
    eval_device = device_override if device_override else device
    model.to(eval_device)
    
    start_time = time.time()
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(eval_device), labels.to(eval_device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    end_time = time.time()
    accuracy = 100 * correct / total
    inference_time = end_time - start_time
    
    print(f'Accuracy: {accuracy:.2f}%')
    print(f'Inference Time: {inference_time:.2f} seconds')
    return accuracy, inference_time

In [8]:
# dataloading
stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_test)
# We only need a small subset for calibration (e.g., 1000 images)
train_subset, _ = torch.utils.data.random_split(trainset, [1000, 49000])
calib_loader = torch.utils.data.DataLoader(train_subset, batch_size=32, shuffle=False, num_workers=2)

In [9]:
float_acc, float_time = test(float_model, testloader)

Accuracy: 95.30%
Inference Time: 1.38 seconds


In [10]:
quant_model = copy.deepcopy(float_model)
quant_model.to('cpu')
quant_model.eval()
print()

In [11]:
quant_model.fuse_model()  # defined in resnet class in resnet_torch.py

torch.backends.quantized.engine = 'fbgemm'
quant_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')

quant_model.quant.qconfig = quant_model.qconfig
quant_model.dequant.qconfig = quant_model.qconfig

# insert observers
torch.quantization.prepare(quant_model, inplace=True)

# calibrate
print("calibrating...")
with torch.no_grad():
    for images, _ in calib_loader:
        quant_model(images)

# convert to int8
torch.quantization.convert(quant_model, inplace=True)
print()

calibrating...


/opt/conda/lib/python3.11/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


In [12]:
print(f"Input Layer Type: {type(quant_model.quant)}")
if isinstance(quant_model.quant, torch.quantization.QuantStub):
    raise RuntimeError("CRITICAL ERROR: self.quant did not convert! It is still a QuantStub.")

Input Layer Type: <class 'torch.ao.nn.quantized.modules.Quantize'>


In [13]:
# measure quantized size
print_size_of_model(quant_model)

Model Size: 11.30 MB


In [14]:
# quantized accuracy
quant_acc, _ = test(quant_model, testloader, device_override=torch.device('cpu'))

Accuracy: 95.27%
Inference Time: 260.53 seconds


In [15]:
float_model.to('cpu')
_, float_cpu_time = test(float_model, testloader, device_override=torch.device('cpu'))

print("Testing Quantized Model on CPU...")
_, quant_cpu_time = test(quant_model, testloader, device_override=torch.device('cpu'))

print(f"\nFloat CPU Time: {float_cpu_time:.2f}s")
print(f"Quant CPU Time: {quant_cpu_time:.2f}s")
print(f"Speedup: {float_cpu_time / quant_cpu_time:.2f}x")

Accuracy: 95.30%
Inference Time: 475.97 seconds
Testing Quantized Model on CPU...
Accuracy: 95.27%
Inference Time: 364.02 seconds

Float CPU Time: 475.97s
Quant CPU Time: 364.02s
Speedup: 1.31x
